## Pulling Papers

In [1]:
# Load Libraries
import os
import shutil
import pandas as pd

# Define the output directory
output_directory = os.path.join(os.getcwd(), "pulling_papers")

# Read the optional PubMed API key
pubmed_api_key_path = os.path.join(os.getcwd(), "../example_data/ncbi_key.txt")
if os.path.exists(pubmed_api_key_path):
    with open(pubmed_api_key_path, "r") as f:
        pubmed_api_key = f.read().strip()
    os.environ["NCBI_API_KEY"] = pubmed_api_key
else:
    pubmed_api_key = None

# Read the scopus api key 
with open(os.path.join(os.getcwd(), "../example_data/scopus_key.txt"), "r" ) as f: 
    scopus_api_key = f.read()

import DancePartner as dance

# Remove it if it already exists and start anew
if os.path.exists(output_directory):
    shutil.rmtree(output_directory)
    os.mkdir(output_directory, mode = 0o777)
else: 
    os.mkdir(output_directory, mode = 0o777)

DancePartner can be used to pull papers from PubMed, Scopus, and OSTI. Papers can be pulled from databases individually, or multiple databases at a time. The default priority is clean text whenever possible followed by titles and abstracts. Users may specify different priorities when pulling text. 

Here we will go through various ways that publications can be pulled. Our examples are:

1. Pulling papers from a single database
2. Pulling papers from multiple databases


### Example 1: Pulling papers from a single database

Here, we have a list of PubMed paper IDs that be either numeric or string type. The last ID does not reference a real paper, but demonstrates how the package acts when a paper is encountered that does not exist.

In [2]:
paper_ids = [9851916, 16803962, 12628183, 15035988, 17626846, 18675916, 21858180, 16803962333333]

Now we call our paper pulling function called `pull_papers`. Note that this function may take a while depending on the number of publications to pull. You should estimate approximately 1-5 seconds per publication.

If you have an NCBI API key saved in `example_data/ncbi_key.txt`, the notebook will pass it to PubMed requests to reduce rate-limit issues.

In [3]:
os.mkdir(os.path.join(output_directory, "pubmed_paper_output_folder"))
dance.pull_papers(
    pubmed_ids = paper_ids,
    output_directory = os.path.join(output_directory, "pubmed_paper_output_folder"),
    pubmed_api_key = pubmed_api_key
)

2026-05-18 11:35:55 WE45748 metapub.findit[86029] INFO FindIt Cache initialized at /Users/degn400/.cache/findit.db


Let's take a look at the content of the folder that we created:

In [5]:
os.listdir(os.path.join(output_directory, "pubmed_paper_output_folder"))

['pubmed_tarballs',
 'pubmed_clean',
 'output_summary.txt',
 'pubmed_abstracts',
 'pubmed_pdfs']

We see that the output folder contains an `output_summary.txt` file as well as several subdirectories. The `pubmed_tarballs` folder contains the `[].tar.gz` files that are used as a subprocess to collect full papers from the PubMed database. They are large files that do not need to be kept. If you would like to write them to a different location, you can specify this in `ppi.pull_papers()` with the optional `tarball_path` parameter. The other three subfolders contain the actual papers themselves. We can glean additional insight into these folders with the `output_summary.txt` file. Let's take a look at what it says:

In [4]:
with open(os.path.join(output_directory, "pubmed_paper_output_folder", "output_summary.txt"), "r") as f:
    print(f.read())

Output Summary for Pulling Papers
Created: 2026-05-18 11:36:01.694426
Total Num. Articles: 8
Total Num. Articles Found: 7
Number of Full Text: 1
Number of Title & Abstracts: 6
Number Missing: 1



This gives a detailed look at the results of the pull_paper function. It shows us when the files were downloaded, the database used, the total number of articles that were searched for, as well as a breakdown of how many papers of each download type were found. Users may also specify which publication type they would like to pull, whether that be "abstracts", "full text", or "both" where priority is given to full text publications. Let's pull some abstracts. 

In [6]:
# Create directory to hold papers 
os.mkdir(os.path.join(output_directory, "pubmed_abstracts"))

# Pull papers
dance.pull_papers(
    pubmed_ids = paper_ids,
    output_directory = os.path.join(output_directory, "pubmed_abstracts"),
    type = "abstract",
    pubmed_api_key = pubmed_api_key,
)

# Read summary file 
with open(os.path.join(output_directory, "pubmed_abstracts", "output_summary.txt"), "r") as f:
    print(f.read())

Output Summary for Pulling Papers
Created: 2026-05-18 11:36:19.187039
Total Num. Articles: 8
Total Num. Articles Found: 7
Number of Full Text: 0
Number of Title & Abstracts: 7
Number Missing: 1



If using the output csv from LitPortal, read the table into python. For PubMed and OSTI, please use the `OriginId` column. For Scopus, please use the `DOI` column. Let's pull papers from scopus. Scopus requires an API key. Save the key as "scopus_key.txt" as put it in your example_data folder. More details can be found here: https://dev.elsevier.com/

In [7]:
# Create directory to hold papers
os.mkdir(os.path.join(output_directory, "scopus_papers"))

# Pull papers
dance.pull_papers(scopus_ids = ["10.1186/s40168-021-01035-8", "10.1002/bit.26296", "10.1002/pmic.200300397", "10.1074/mcp.M115.057117"],
                output_directory = os.path.join(output_directory, "scopus_papers"), scopus_api_key = scopus_api_key)

# Read summary file 
with open(os.path.join(output_directory, "scopus_papers", "output_summary.txt"), "r") as f:
    print(f.read())

Output Summary for Pulling Papers
Created: 2026-05-18 11:36:25.681155
Total Num. Articles: 4
Total Num. Articles Found: 4
Number of Full Text: 1
Number of Title & Abstracts: 3
Number Missing: 0



And finally, here is an example using OSTI. 

In [8]:
# Create directory to hold osti papers
os.mkdir(os.path.join(output_directory, "osti_papers"))

# Pull papers
dance.pull_papers(osti_ids = ["2229172", "1629838", "1766618", "1379914"], output_directory = os.path.join(output_directory, "osti_papers"))

# Read summary file 
with open(os.path.join(output_directory, "osti_papers", "output_summary.txt"), "r") as f:
    print(f.read())

Output Summary for Pulling Papers
Created: 2026-05-18 11:36:31.158096
Total Num. Articles: 4
Total Num. Articles Found: 1
Number of Full Text: 0
Number of Title & Abstracts: 1
Number Missing: 3



### Example 2: Extract Paper IDs from a Query

In [9]:
# Pull the first 100 matching papers from PubMed - extracting from the years 2000 to 2025
pmed_query = dance.query_pubmed(
    query = '(e coli proteomics) AND (e coli metabolism) AND ("2000/01/01"[Date - Publication]:"2025/12/31"[Date - Publication])', 
    pubmed_api_key = pubmed_api_key,
    max_results = 100
)
pmed_query

,PMID,Title,DOI
0,41597609,Bactericidal Activity of Selenium Nanoparticle...,10.3390/microorganisms14010089
1,41480317,Combined metabolomic and metagenomic analysis ...,10.3748/wjg.v31.i48.112653
2,41478473,Valorisation of gilthead seabream by-products ...,10.1016/j.ijbiomac.2025.150014
3,41478466,Design and application of a multi-epitope ACE2...,10.1016/j.imlet.2025.107127
4,41477766,Aromatic microbial metabolite hippuric acid en...,10.1016/j.celrep.2025.116749
...,...,...,...
95,40792504,Novel enzymes involved in the biotransformatio...,10.1128/spectrum.00562-25
96,40764731,Molecular self-assembly mediates the flocculat...,10.1038/s41598-025-13837-z
97,40762489,A large-scale screening campaign of putative c...,10.1128/mbio.01007-25
98,40733845,Mass spectrometric analysis of lipid-linked ol...,10.1016/j.carbpol.2025.123928


In [10]:
scopus_query = dance.query_scopus(
    query = '( TITLE-ABS-KEY ( e coli proteomics ) AND TITLE-ABS-KEY ( e coli metabolomics ) ) AND PUBYEAR > 1999 AND PUBYEAR < 2026',
    scopus_api_key = scopus_api_key,
    max_results = 100
)
scopus_query

,Title,DOI,EID
0,Active and molecular biological mechanisms of ...,10.1016/j.jece.2025.120472,2-s2.0-105023587373
1,Biochemical characteristics of extracts from p...,10.1186/s12864-025-11862-w,2-s2.0-105013568974
2,Wastewater-based surveillance reveals distinct...,10.1016/j.envpol.2025.126950,2-s2.0-105012601389
3,"From Infection to Infertility: Diagnostic, The...",10.3390/ani15192841,2-s2.0-105019236159
4,Human gut bacteria bioaccumulate per- and poly...,10.1038/s41564-025-02032-5,2-s2.0-105009546819
...,...,...,...
91,Multi-omics data-driven systems biology of E. ...,10.1007/978-1-4020-9394-4_3,2-s2.0-84900901782
92,Evaluation of affinity-tagged protein expressi...,10.1021/pr801088f,2-s2.0-67650382230
93,Growth phase-dependent global protein and meta...,10.1002/pmic.200900120,2-s2.0-68049109520
94,A novel data mining method to identify assay-s...,10.1186/1471-2105-7-377,2-s2.0-33749601618


In [11]:
osti_query = dance.query_osti('"e coli proteomics" AND "e coli metabolism" AND publication_date:[2000-01-01 TO 2025-12-31]', max_results = 100)
osti_query

,TITLE,DOI,OSTI_IDENTIFIER
0,Final project report,10.2172/3014296,3014296
1,Protein–Protein Interaction Networks Derived f...,10.1021/acs.jproteome.4c00535,2483344
2,Recent developments in enzymatic and microbial...,10.1016/j.jbiotec.2024.04.004,2576730
3,Comparative studies of glycolytic pathways and...,10.1002/aic.16367,1472164
4,SteadyCom: Predicting microbial abundances whi...,10.1371/journal.pcbi.1005539,1360660
5,A Comparison of the Costs and Benefits of Bact...,10.1371/journal.pone.0164314,1338429
6,Gene fusions and gene duplications: relevance ...,10.1186/1471-2164-6-33,1626445
7,Tools for the Microbiome: Nano and Beyond,10.1021/acsnano.5b07826,1233975
8,"Proceedings of Synthetic Biology: Engineering,...",,1330220


In [12]:
# Optionally, deduplicate the results -- more below
dance.deduplicate_papers(
    pubmed_path = pmed_query,
    scopus_path = scopus_query,
    osti_path = osti_query
)

,Title,DOI,pubmed,scopus,osti
0,a comparison of the costs and benefits of bact...,10.1371/journal.pone.0164314,NaN,NaN,1338429
1,a dualmechanism antimicrobial peptide with ant...,10.1128/msphere.00068-25,40879398,NaN,NaN
2,a flexible framework for sparse simultaneous c...,10.1186/1471-2105-12-448,NaN,2-s2.0-81055127037,NaN
3,a largescale screening campaign of putative ca...,10.1128/mbio.01007-25,40762489,NaN,NaN
4,a microbiotaderived bile acid modulates biofil...,10.1038/s41522-025-00854-z,41345396,NaN,NaN
...,...,...,...,...,...
199,valorisation of gilthead seabream byproducts t...,10.1016/j.ijbiomac.2025.150014,41478473,NaN,NaN
200,wastewaterbased surveillance reveals distinct ...,10.1016/j.envpol.2025.126950,NaN,2-s2.0-105012601389,NaN
201,what worth the garlic peel,10.3390/ijms23042126,NaN,2-s2.0-85124476226,NaN
202,yeast growth is controlled by the proportional...,,40895079,NaN,NaN


### Example 3: Pulling Papers from Multiple Databases

Oftentimes, we want to pull papers from more than just one database. To do so, we pass a different set of arguments to our `pull_papers` function. Instead of specifying a database and a list of IDs, we can instead feed strings pointing to the CSV files downloaded from each database.

In [13]:
pubmed_path = os.path.join(os.getcwd(), "vignette_data/PubMed_Export.csv")
scopus_path = os.path.join(os.getcwd(), "vignette_data/Scopus_Export.csv")
osti_path = os.path.join(os.getcwd(), "vignette_data/OSTI_Export.csv")

First, let's deduplicate the papers with the deduplicate_papers function.

We will now specify the output folder location as we did previously. Please note that using Scopus requires an API key to function properly. Instructions to obtain one can be found [here](https://dev.elsevier.com/). We will read in our key here, so remember to replace these lines with yours.

In [14]:
deduplicated_papers = dance.deduplicate_papers(pubmed_path, scopus_path, osti_path)
deduplicated_papers

,Title,DOI,pubmed,scopus,osti
0,12th international mouse genome conference,NaN,NaN,NaN,760867.0
1,13c and 15nlabeling strategies combined with m...,10.1371/journal.pone.0141850,26528916.0,NaN,NaN
2,2004 environmental mutagen society annual meet...,10.1002/em.20057,NaN,NaN,877190.0
3,2016 national algal biofuels technology review,10.2172/1259407,NaN,NaN,1259407.0
4,2nd international conference on pathways netwo...,NaN,NaN,NaN,860359.0
...,...,...,...,...,...
1694,zn deficiency disrupts cu and s homeostasis in...,10.1093/mtomcs/mfad043,NaN,NaN,2326175.0
1695,zyxin contributes to coupling between cell jun...,10.1371/journal.pgen.1010319,36976799.0,NaN,NaN
1696,βintegrin dephosphorylation by the densityenha...,10.1371/journal.pgen.1006592,28135265.0,NaN,NaN
1697,μmapred proximity labeling by red light photoc...,10.1021/jacs.2c01384,NaN,NaN,1978471.0


This is the deduped_table that is needed by pull_papers(). Papers will be pulled prioritizing full text to abstracts, in the order of pubmed, scopus, and OSTI. 

In [15]:
output_directory = os.path.join(os.getcwd(), "pulling_papers")

# Make an example for this deduplicated data 
os.mkdir(os.path.join(output_directory, "deduped_example"))

# Read the scopus api key 
with open(os.path.join(os.getcwd(), "../example_data/scopus_key.txt"), "r" ) as f: 
    scopus_api_key = f.read()

# To save time, let's do a subset of the deduplicated papers
a_subset = pd.concat([deduplicated_papers.head(10), deduplicated_papers.tail(10)]).reset_index(drop = True)

# Pull the papers. To save time, let's do the first 10 rows and the last 10 rows
dance.pull_papers(
    deduped_table = a_subset,
    output_directory = os.path.join(output_directory, "deduped_example"),
    pubmed_api_key = pubmed_api_key,
    scopus_api_key = scopus_api_key,
)

# Read summary file 
with open(os.path.join(output_directory, "deduped_example", "output_summary.txt"), "r") as f:
    print(f.read())

Output Summary for Pulling Papers
Created: 2026-05-18 11:37:02.524180
Total Num. Articles: 20
Total Num. Articles Found: 12
Number of Full Text: 0
Number of Title & Abstracts: 12
Number Missing: 8



A note on adding publications. They must be added as txt files. The `pypdf` package can be used to convert a pdf to txt file using the `pdfReader()`. Example code is below

```{python}
# Load library
import pypdf

# Read data
reader = PdfReader("your_file.pdf")

# Hold text
text = []

with open("your_file.txt", "w") as file:
    for page in reader.pages:
        file.write(page.extract_text() + "\n")
```